In [1]:
"""
Student Result Processor
-------------------------
Computes per-student and per-subject statistics, ranks students and
subjects, lets the user look up a subject, and prints/saves a clean
results report.

Data is loaded directly in the code below (no external file needed).
Pure Python - no pandas, no classes.
"""

REPORT_FILE = "report.txt"

# ---------------------------------------------------------------------
# DATA (loaded directly here — name + each subject score)
# ---------------------------------------------------------------------

SUBJECTS = ["Mathematics", "English", "Physics", "Chemistry", "Biology"]

RAW_SCORES = [
    ("Chidi Okafor",     [71, 72, 69, 70, 73]),
    ("Amina Bello",      [90, 91, 88, 88, 91]),
    ("Tunde Adeyemi",    [47, 48, 44, 45, 47]),
    ("Ngozi Eze",        [70, 71, 67, 68, 70]),
    ("Ibrahim Musa",     [82, 83, 80, 80, 83]),
    ("Blessing Effiong", [58, 60, 56, 56, 59]),
    ("Emeka Nwankwo",    [42, 43, 40, 40, 43]),
    ("Fatima Sani",      [74, 75, 71, 71, 74]),
]


def build_students(raw_scores, subjects):
    """
    Turn the raw (name, [scores]) data above into a list of dictionaries,
    one dict per student, keyed by column name — same shape you'd get
    from reading a CSV. Any score that is missing or not a valid number
    is skipped for that student rather than crashing the program.
    """
    students = []
    for name, scores in raw_scores:
        student = {"name": name}
        for subject, value in zip(subjects, scores):
            if value is None or value == "":
                continue
            try:
                student[subject] = float(value)
            except (TypeError, ValueError):
                continue
        students.append(student)
    return students


# ---------------------------------------------------------------------
# PER-STUDENT AVERAGE & GRADE
# ---------------------------------------------------------------------

def get_grade(average):
    """Return a letter grade for a given average, using the fixed scale."""
    if average >= 70:
        return "A"
    elif average >= 60:
        return "B"
    elif average >= 50:
        return "C"
    elif average >= 45:
        return "D"
    else:
        return "F"


def compute_student_average(student, subjects):
    scores = [student[subj] for subj in subjects if subj in student]
    if not scores:
        return None
    return sum(scores) / len(scores)


def compute_student_stats(students, subjects):
    stats = []
    for student in students:
        avg = compute_student_average(student, subjects)
        if avg is None:
            continue
        stats.append((student["name"], avg, get_grade(avg)))
    return stats


# ---------------------------------------------------------------------
# CLASS AVERAGE PER SUBJECT
# ---------------------------------------------------------------------

def compute_subject_averages(students, subjects):
    subject_averages = {}
    for subject in subjects:
        scores = [s[subject] for s in students if subject in s]
        subject_averages[subject] = sum(scores) / len(scores) if scores else 0.0
    return subject_averages


# ---------------------------------------------------------------------
# TOP STUDENT
# ---------------------------------------------------------------------

def find_top_student(student_stats):
    return max(student_stats, key=lambda entry: entry[1])


# ---------------------------------------------------------------------
# PASS / FAIL COUNT
# ---------------------------------------------------------------------

def count_pass_fail(student_stats, pass_mark=50):
    passed = sum(1 for _, avg, _ in student_stats if avg >= pass_mark)
    failed = len(student_stats) - passed
    return passed, failed


# ---------------------------------------------------------------------
# RANKED LEADERBOARD
# ---------------------------------------------------------------------

def rank_students(student_stats):
    return sorted(student_stats, key=lambda entry: entry[1], reverse=True)


def rank_subjects_hardest_first(subject_averages):
    return sorted(subject_averages.items(), key=lambda item: item[1])


# ---------------------------------------------------------------------
# OVERALL CLASS AVERAGE
# ---------------------------------------------------------------------

def compute_overall_average(students, subjects):
    all_scores = [s[subject] for s in students for subject in subjects if subject in s]
    if not all_scores:
        return 0.0
    return sum(all_scores) / len(all_scores)


# ---------------------------------------------------------------------
# SUBJECT LOOKUP
# ---------------------------------------------------------------------

def subject_summary(students, subject):
    entries = [(s["name"], s[subject]) for s in students if subject in s]
    if not entries:
        return None
    avg = sum(score for _, score in entries) / len(entries)
    highest_student, highest_score = max(entries, key=lambda e: e[1])
    lowest_student, lowest_score = min(entries, key=lambda e: e[1])
    return avg, highest_score, highest_student, lowest_score, lowest_student


def ask_subject_lookup(students, subjects):
    print("\nAvailable subjects: " + ", ".join(subjects))
    typed = input("Type a subject name to see its summary: ").strip()

    match = next((s for s in subjects if s.lower() == typed.lower()), None)
    if match is None:
        message = f"'{typed}' is not a recognised subject.\n"
        print(message)
        return message

    result = subject_summary(students, match)
    if result is None:
        message = f"No valid scores recorded for {match}.\n"
        print(message)
        return message

    avg, high_score, high_name, low_score, low_name = result
    lines = [
        f"\n--- {match} Summary ---",
        f"Class average : {avg:.2f}",
        f"Highest score : {high_score:.1f} ({high_name})",
        f"Lowest score  : {low_score:.1f} ({low_name})",
    ]
    output = "\n".join(lines) + "\n"
    print(output)
    return output


# ---------------------------------------------------------------------
# REPORT GENERATION (formatted as an "answer key")
# ---------------------------------------------------------------------

def build_report(students, subjects):
    student_stats = compute_student_stats(students, subjects)
    subject_averages = compute_subject_averages(students, subjects)
    top_name, top_avg, top_grade = find_top_student(student_stats)
    passed, failed = count_pass_fail(student_stats)
    ranked_students = rank_students(student_stats)
    ranked_subjects = rank_subjects_hardest_first(subject_averages)
    overall_avg = compute_overall_average(students, subjects)

    lines = []
    lines.append("STUDENT RESULT PROCESSOR — ANSWER KEY (extended)")
    lines.append("=" * 54)

    lines.append("\n1) PER-STUDENT AVERAGE & GRADE\n")
    for name, avg, grade in student_stats:
        lines.append(f"   {name:<20} avg={avg:>6.2f}  grade={grade}")

    lines.append("\n2) CLASS AVERAGE PER SUBJECT\n")
    for subject in subjects:
        lines.append(f"   {subject:<15} class avg = {subject_averages[subject]:.2f}")

    lines.append("\n3) TOP STUDENT (highest overall average)\n")
    lines.append(f"   {top_name}  ({top_avg:.2f})")

    lines.append("\n4) PASS / FAIL COUNT (pass = overall avg >= 50)\n")
    lines.append(f"   Passed: {passed}    Failed: {failed}")

    lines.append("\n5) STUDENT LEADERBOARD (highest to lowest average)\n")
    for rank, (name, avg, grade) in enumerate(ranked_students, start=1):
        lines.append(f"   {rank}. {name:<20}{avg:>7.2f}  ({grade})")

    lines.append("\n6) SUBJECTS RANKED HARDEST TO EASIEST (lowest avg = hardest)\n")
    n = len(ranked_subjects)
    for rank, (subject, avg) in enumerate(ranked_subjects, start=1):
        tag = "  <- hardest" if rank == 1 else ("  <- easiest" if rank == n else "")
        lines.append(f"   {rank}. {subject:<15}{avg:>7.2f}{tag}")

    lines.append(f"\n7) OVERALL CLASS AVERAGE (all scores): {overall_avg:.2f}")

    return "\n".join(lines)


def save_report(report_text, filename=REPORT_FILE):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report_text)


# ---------------------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------------------

def main():
    students = build_students(RAW_SCORES, SUBJECTS)

    if not students:
        print("No valid student data found.")
        return

    report_text = build_report(students, SUBJECTS)
    print(report_text)

    lookup_text = ask_subject_lookup(students, SUBJECTS)

    full_report = report_text + "\n" + lookup_text
    save_report(full_report)
    print(f"Report saved to {REPORT_FILE}")


if __name__ == "__main__":
    main()

STUDENT RESULT PROCESSOR — ANSWER KEY (extended)

1) PER-STUDENT AVERAGE & GRADE

   Chidi Okafor         avg= 71.00  grade=A
   Amina Bello          avg= 89.60  grade=A
   Tunde Adeyemi        avg= 46.20  grade=D
   Ngozi Eze            avg= 69.20  grade=B
   Ibrahim Musa         avg= 81.60  grade=A
   Blessing Effiong     avg= 57.80  grade=C
   Emeka Nwankwo        avg= 41.60  grade=F
   Fatima Sani          avg= 73.00  grade=A

2) CLASS AVERAGE PER SUBJECT

   Mathematics     class avg = 66.75
   English         class avg = 67.88
   Physics         class avg = 64.38
   Chemistry       class avg = 64.75
   Biology         class avg = 67.50

3) TOP STUDENT (highest overall average)

   Amina Bello  (89.60)

4) PASS / FAIL COUNT (pass = overall avg >= 50)

   Passed: 6    Failed: 2

5) STUDENT LEADERBOARD (highest to lowest average)

   1. Amina Bello           89.60  (A)
   2. Ibrahim Musa          81.60  (A)
   3. Fatima Sani           73.00  (A)
   4. Chidi Okafor          71.00  (A

Type a subject name to see its summary:  English



--- English Summary ---
Class average : 67.88
Highest score : 91.0 (Amina Bello)
Lowest score  : 43.0 (Emeka Nwankwo)

Report saved to report.txt
